# GenAI Assignment 1: training notebook (Colab / Kaggle)
Runs the full pipeline on a free GPU: data prep -> Tasks 1-4 (Optuna + MLflow) -> evaluation -> ONNX export.
**Runtime > Change runtime type > GPU (T4)** before you start. Budgets below are for a T4; each task writes checkpoints to Google Drive so a disconnect does not lose work.

## 0. Mount Drive and get the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/genai_a1'          # checkpoints, studies, mlflow.db and outputs live here
os.makedirs(WORK, exist_ok=True)

In [ ]:
# Option A: clone your GitHub repo (replace the URL)
REPO_URL = 'https://github.com/<your-user>/<your-repo>.git'
!git clone $REPO_URL /content/repo || (cd /content/repo && git pull)
# Option B: upload GenAI_Assignment1.zip to Drive and unzip:  !unzip -q $WORK/GenAI_Assignment1.zip -d /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

In [ ]:
# Keep everything that must survive a disconnect on Drive by symlinking the working folders there
for d in ['checkpoints','studies','outputs','mlartifacts','models','data']:
    os.makedirs(f'{WORK}/{d}', exist_ok=True)
    if not os.path.islink(d):
        !rm -rf $d
        os.symlink(f'{WORK}/{d}', d)
if not os.path.exists(f'{WORK}/mlflow.db'): open(f'{WORK}/mlflow.db','a').close()
if not os.path.islink('mlflow.db'): os.symlink(f'{WORK}/mlflow.db','mlflow.db')
os.environ['PYTHONPATH']='src'
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable a GPU runtime')

## 1. Data
**Pets** downloads automatically. **FS2K** must be downloaded by you from the dataset page (github.com/DengPingFan/FS2K, Google Drive link in its README).
Put the extracted `FS2K` folder (containing `photo/`, `sketch/`, `anno_train.json`, `anno_test.json`) at `genai_a1/data/FS2K` on your Drive.

In [ ]:
!python -m genai.prepare --pets

In [ ]:
assert os.path.exists('data/FS2K/anno_train.json'), 'Upload FS2K to MyDrive/genai_a1/data/FS2K first'
!python -m genai.prepare --fs2k

## 2. Task 1: universal autoencoder (Optuna study, then final training)

In [ ]:
!python -m genai.task1 --mode all --trials 25 --trial-epochs 6 --final-epochs 60

## 3. Task 2: classifier, then three specialists

In [ ]:
!python -m genai.task2 --part classifier --mode all --trials 20 --trial-epochs 5 --final-epochs 40

In [ ]:
!python -m genai.task2 --part specialists --mode all --trials 15 --trial-epochs 4 --final-epochs 40

## 4. Task 3: soft mixture-of-experts (needs Task 2 checkpoints)

In [ ]:
!python -m genai.task3 --mode all --trials 20 --trial-epochs 4 --warmup-epochs 2 --final-epochs 30

## 5. Task 4: conditional GAN

In [ ]:
!python -m genai.task4 --mode all --trials 15 --trial-epochs 10 --final-epochs 150 --sample-every 10

## 6. Evaluation on the held-out test sets, ONNX export, app samples

In [ ]:
!python -m genai.evaluate --task all
!python -m genai.export
!python scripts/make_samples.py
!python scripts/make_diagrams.py
!python -m genai.report_assets

## 7. Look at the experiment tracking and package the results

In [ ]:
import json; print(json.dumps(json.load(open('outputs/onnx_parity.json')), indent=1)[:1500])

In [ ]:
# Zip what you need locally: ONNX models, results, figures, optuna summaries, report assets, samples
!cd /content/repo && zip -qr $WORK/results_bundle.zip models outputs/results outputs/figures outputs/optuna outputs/*.json report/generated report/figures app/backend/samples mlflow.db
print('Download', f'{WORK}/results_bundle.zip', 'from Drive (right-click > Download)')

### MLflow UI screenshot for the report
Run locally after downloading the bundle: `mlflow ui --backend-store-uri sqlite:///mlflow.db` and screenshot http://127.0.0.1:5000
(or run it here with `!pip install pyngrok`; the local way is simpler).